# 03 — Live experiment against the deployed MSK cluster

Unlike `00`-`02`, which point at `$FDAI_TARGET` (local by default), this one is
pinned to the real cloud cluster on purpose — there is no ambiguity about which
broker you are looking at. A few small, quick experiments on whatever is
actually flowing right now, not a full analysis (see `01_explore_trades.ipynb`
for that).

**Prerequisites:** `make up` has succeeded, and this environment has the same
AWS credentials active that `terraform` used — `from_terraform()` shells out to
`terraform output` under the hood.

## If AWS credentials have expired

Sandbox SSO sessions commonly expire in hours — well before the account's
7-day wipe. If the next cell raises `TargetError: AWS credentials are
invalid or expired...`, run this first, then re-run it — no need to
restart Jupyter (see `notebooks/README.md` Troubleshooting).

`fdai-sandbox` is the example profile name from `docs/SETUP.md` §2;
replace it if yours differs. Skip this cell entirely if your org issues
raw/temporary keys instead of SSO.

In [12]:
!aws sso login --profile fdai-sandbox

Attempting to open your default browser. If the browser does not open, open the following URL.
If you are unable to open the URL on this device, run this command again with the '--use-device-code' option.

https://oidc.eu-central-1.amazonaws.com/authorize?response_type=code&client_id=a0YtZYShATkIdsWaPYtBNWV1LWNlbnRyYWwtMQ&redirect_uri=http%3A%2F%2F127.0.0.1%3A63735%2Foauth%2Fcallback&state=b52f5b2a-e8ef-4df8-a92a-7a3a37998a18&code_challenge_method=S256&scopes=sso%3Aaccount%3Aaccess&code_challenge=X0dUCBlQuX9GfFtKmRdW8sMIioD1hpWO8lwliOmaUC0

Successfully logged into Start URL: https://identitycenter.amazonaws.com/ssoins-69871294d6700676


In [2]:
%env AWS_PROFILE=fdai-sandbox

env: AWS_PROFILE=fdai-sandbox


In [14]:
import devlab
from devlab import frames

target = devlab.from_terraform()  # the deployed MSK cluster, explicitly — not $FDAI_TARGET
target

Target(name='msk', bootstrap='\x1b╷\x1b\x1b\n\x1b│\x1b \x1b\x1b\x1bWarning: \x1b\x1b\x1bNo outputs found\x1b\n\x1b│\x1b \x1b\n\x1b│\x1b \x1b\x1bThe state file either has no outputs defined, or all the defined outputs\n\x1b│\x1b \x1bare empty. Please define an output in your configuration with the `output`\n\x1b│\x1b \x1bkeyword and run `terraform refresh` for it to become available. If you are\n\x1b│\x1b \x1busing interpolation, please verify the interpolated value is not empty. You\n\x1b│\x1b \x1bcan use the `terraform console` command to assist.\n\x1b╵\x1b\x1b', sasl_username='\x1b╷\x1b\x1b\n\x1b│\x1b \x1b\x1b\x1bWarning: \x1b\x1b\x1bNo outputs found\x1b\n\x1b│\x1b \x1b\n\x1b│\x1b \x1b\x1bThe state file either has no outputs defined, or all the defined outputs\n\x1b│\x1b \x1bare empty. Please define an output in your configuration with the `output`\n\x1b│\x1b \x1bkeyword and run `terraform refresh` for it to become available. If you are\n\x1b│\x1b \x1busing interpolation, please veri

## Is it actually live right now

Cheap, and worth doing before anything else: reads from `latest` for 10
seconds. Zero here means the producer host isn't running — check that before
assuming anything below is broken.

In [15]:
report = devlab.rate(target, seconds=10.0)
print(f"{report.messages} trades in {report.seconds:.1f}s = {report.per_second:.1f}/s")
report.by_venue

%3|1786290128.133|FAIL|rdkafka#consumer-1| [thrd:sasl_ssl://╷
│ Warning:0]: sasl_ssl://╷
│ Warning:0/bootstrap: Failed to resolve '╷
│ Warning:0': nodename nor servname provided, or not known (after 55ms in state CONNECT)
%3|1786290128.135|FAIL|rdkafka#consumer-1| [thrd:sasl_ssl://or all the defined outputs
│ are empt]: sasl_ssl://or all the defined outputs
│ are empty. Please define an output in your configuration with the `output`
│ keyword and run `terraform refresh` for it to become available. If you are
│ using interpolat: Failed to resolve 'or all the defined outputs
│ are empty. Please define an output in your configuration with the `output`
│ keyword and run `terraform refresh` for it to become available. If you are
│ using interpolation:9092': nodename nor servname provided, or not known (after 1ms in state CONNECT)
%3|1786290128.189|FAIL|rdkafka#consumer-1| [thrd:sasl_ssl://please verify the interpolated value is not empty. Y]: sasl_ssl://please verify the interpolated value 

BrokerUnavailable: could not reach the broker at '\x1b[33m╷\x1b[0m\x1b[0m\n\x1b[33m│\x1b[0m \x1b[0m\x1b[1m\x1b[33mWarning: \x1b[0m\x1b[0m\x1b[1mNo outputs found\x1b[0m\n\x1b[33m│\x1b[0m \x1b[0m\n\x1b[33m│\x1b[0m \x1b[0m\x1b[0mThe state file either has no outputs defined, or all the defined outputs\n\x1b[33m│\x1b[0m \x1b[0mare empty. Please define an output in your configuration with the `output`\n\x1b[33m│\x1b[0m \x1b[0mkeyword and run `terraform refresh` for it to become available. If you are\n\x1b[33m│\x1b[0m \x1b[0musing interpolation, please verify the interpolated value is not empty. You\n\x1b[33m│\x1b[0m \x1b[0mcan use the `terraform console` command to assist.\n\x1b[33m╵\x1b[0m\x1b[0m' (target 'msk') before the topic could be checked: KafkaError{code=_TRANSPORT,val=-195,str="Failed to get metadata: Local: Broker transport failure"}. For public MSK, verify that the endpoint is current with `devlab.from_terraform()` and that this machine's current public IP is allowlisted; after r

## Grab a batch of real trades

Bounded by both a count and a clock, so this returns even against a quiet
topic. `offset_reset="earliest"` reads what's already retained rather than
only what arrives from this point on.

In [ ]:
records = devlab.collect(target, limit=2_000, seconds=30.0, offset_reset="earliest")
df = frames.trades_frame(records)
print(f"{len(df):,} trades  {df['event_ts'].min()} .. {df['event_ts'].max()}")
df.head()

## Experiment 1 — who's trading what

Per venue and instrument: trade count, volume, and notional. Sorted by
notional, so whatever's actually moving money floats to the top.

In [ ]:
(
    df.groupby(["venue", "instrument_id"], observed=True)
    .agg(trades=("trade_id", "count"), volume=("size", "sum"), notional=("notional", "sum"))
    .sort_values("notional", ascending=False)
)

## Experiment 2 — price, live

Whichever instrument traded the most in this window, plotted as-is (no
resampling) — the rawest possible look at the tape.

In [ ]:
top_instrument = df["instrument_id"].value_counts().idxmax()
subset = df[df["instrument_id"] == top_instrument]
axis = subset.plot(x="event_ts", y="price", figsize=(10, 3), title=f"{top_instrument} — live")
axis.set_xlabel("event time (UTC)")

## Experiment 3 — a quick VWAP

Dedupe first (a repaired trade can appear twice — see `02_prototype_silver.ipynb`
for why), then 1-minute bars. Just the tail, since this is meant to be a quick
look, not the full analysis.

In [ ]:
bars = frames.bars(frames.dedupe(df), freq="1min")
bars.tail()